# 06. Model Comparison and Final Forecast

В этом ноутбуке объединяем метрики моделей, выбираем лучшую модель по WAPE и строим финальный прогноз на 30 дней вперед.

## 1. Load model metrics

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    Prophet = None
    PROPHET_AVAILABLE = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualization import save_plot

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
IMAGES.mkdir(parents=True, exist_ok=True)

arima_metrics_path = DATA_PROCESSED / "arima_metrics.csv"
if not arima_metrics_path.exists():
    raise FileNotFoundError("Run notebooks/04_arima_model.ipynb first.")

metrics_frames = [pd.read_csv(arima_metrics_path)]

prophet_metrics_path = DATA_PROCESSED / "prophet_metrics.csv"
if prophet_metrics_path.exists():
    metrics_frames.append(pd.read_csv(prophet_metrics_path))
else:
    print("Prophet metrics file was not found. Comparison will use available models.")

metrics = pd.concat(metrics_frames, ignore_index=True)
display(metrics)

## 2. Compare baseline, SARIMA and Prophet

In [ ]:
comparison = metrics.sort_values("WAPE", ascending=True).reset_index(drop=True)
comparison_path = DATA_PROCESSED / "model_comparison.csv"
comparison.to_csv(comparison_path, index=False)

display(comparison)
print(f"Saved comparison to: {comparison_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(comparison["model"], comparison["WAPE"])
ax.set_title("Model Comparison by WAPE")
ax.set_xlabel("model")
ax.set_ylabel("WAPE, %")
ax.grid(True)
plt.xticks(rotation=30, ha="right")
save_plot(IMAGES / "model_comparison_wape.png")
plt.show()

## 3. Select best model by WAPE

In [ ]:
best_model = comparison.loc[0, "model"]
best_wape = comparison.loc[0, "WAPE"]

print(f"Best model by WAPE: {best_model}")
print(f"Best WAPE: {best_wape:.2f}%")

## 4. Forecast next 30 days

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "daily_sales_series.csv", parse_dates=["date"])
target_df = df[["date", "sales_clean"]].rename(columns={"sales_clean": "sales"}).copy()
future_dates = pd.date_range(target_df["date"].max() + pd.Timedelta(days=1), periods=30, freq="D")


def forecast_with_sarima(full_df, dates):
    full_series = full_df.set_index("date")["sales"].asfreq("D")
    model = SARIMAX(
        full_series,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit(disp=False, maxiter=100)
    forecast_values = result.get_forecast(steps=len(dates)).predicted_mean
    return pd.DataFrame({"date": dates, "forecast": np.maximum(forecast_values.to_numpy(), 0), "model": "SARIMA"})


def forecast_with_baseline(full_df, dates):
    forecast_value = full_df["sales"].tail(7).mean()
    return pd.DataFrame({"date": dates, "forecast": forecast_value, "model": "Baseline 7-day mean"})


def prepare_future_regressors(history_df, dates, regressors):
    future = pd.DataFrame({"ds": dates})
    for regressor in regressors:
        if regressor == "onpromotion":
            future[regressor] = 0
        elif regressor == "is_holiday":
            future[regressor] = 0
        elif regressor == "dcoilwtico":
            last_value = history_df[regressor].ffill().bfill().iloc[-1]
            future[regressor] = 0 if pd.isna(last_value) else last_value
        elif regressor == "transactions":
            rolling_average = history_df[regressor].tail(30).mean()
            future[regressor] = 0 if pd.isna(rolling_average) else rolling_average
        else:
            last_value = history_df[regressor].ffill().bfill().iloc[-1]
            future[regressor] = 0 if pd.isna(last_value) else last_value
    return future


def forecast_with_prophet(history_df, dates):
    regressor_candidates = ["onpromotion", "is_holiday", "transactions", "dcoilwtico"]
    regressors = [col for col in regressor_candidates if col in history_df.columns]
    prophet_history = history_df[["date", "sales_clean"] + regressors].rename(
        columns={"date": "ds", "sales_clean": "y"}
    ).copy()
    for regressor in regressors:
        if regressor == "dcoilwtico":
            prophet_history[regressor] = prophet_history[regressor].ffill().bfill()
        else:
            prophet_history[regressor] = prophet_history[regressor].fillna(0)
    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    for regressor in regressors:
        model.add_regressor(regressor)
    model.fit(prophet_history)
    future = prepare_future_regressors(history_df, dates, regressors)
    forecast = model.predict(future)
    return pd.DataFrame({"date": dates, "forecast": np.maximum(forecast["yhat"].to_numpy(), 0), "model": "Prophet"})


if "Prophet" in str(best_model):
    if PROPHET_AVAILABLE:
        final_forecast = forecast_with_prophet(df, future_dates)
    else:
        print("Prophet is not installed. Falling back to SARIMA for the final forecast.")
        final_forecast = forecast_with_sarima(target_df, future_dates)
elif "SARIMA" in str(best_model):
    final_forecast = forecast_with_sarima(target_df, future_dates)
elif "Baseline" in str(best_model):
    final_forecast = forecast_with_baseline(target_df, future_dates)
else:
    final_forecast = forecast_with_sarima(target_df, future_dates)

display(final_forecast.head())

## 5. Business interpretation

In [ ]:
forecast_total = final_forecast["forecast"].sum()
forecast_avg = final_forecast["forecast"].mean()
forecast_min = final_forecast["forecast"].min()
forecast_max = final_forecast["forecast"].max()

print(f"Final model used: {final_forecast['model'].iloc[0]}")
print(f"Forecast horizon: {final_forecast['date'].min().date()} - {final_forecast['date'].max().date()}")
print(f"Total forecasted demand for 30 days: {forecast_total:,.2f}")
print(f"Average daily forecasted demand: {forecast_avg:,.2f}")
print(f"Min daily forecast: {forecast_min:,.2f}")
print(f"Max daily forecast: {forecast_max:,.2f}")

print("\nBusiness use:")
print("- use total 30-day forecast as an input for purchase planning;")
print("- use daily forecast to plan replenishment and safety stock;")
print("- compare promo calendar with expected demand peaks;")
print("- monitor WAPE because it shows error relative to total sales volume.")

## 6. Save final outputs

In [ ]:
final_forecast_path = DATA_PROCESSED / "final_30_days_forecast.csv"
final_forecast.to_csv(final_forecast_path, index=False)
print(f"Saved final forecast to: {final_forecast_path}")

fig, ax = plt.subplots(figsize=(12, 5))
history_tail = target_df.tail(180)
ax.plot(history_tail["date"], history_tail["sales"], label="history")
ax.plot(final_forecast["date"], final_forecast["forecast"], label="final forecast")
ax.set_title("Final 30-Day Demand Forecast")
ax.set_xlabel("date")
ax.set_ylabel("sales")
ax.legend()
ax.grid(True)
save_plot(IMAGES / "final_forecast.png")
plt.show()

## Business interpretation

Финальный прогноз можно использовать как основу для закупок и планирования запасов на ближайший месяц. Суммарный прогноз за 30 дней помогает оценить общий объем закупки, а дневной прогноз - распределить пополнение по календарю. Если ожидаются промо или праздники, прогноз нужно сверять с промо-календарем и доступностью товара.

Ограничения: проект демонстрационный, анализирует один ряд, использует один holdout-период и не знает точные будущие значения части внешних факторов. WAPE удобен для продаж, потому что показывает ошибку относительно общего объема спроса и лучше интерпретируется бизнесом, чем абсолютная ошибка без масштаба.